### add the phase_amplitude to phasenet & EQT results

In [19]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# === 1️⃣ 读取两个文件（修改为你的路径） ===
df1 = pd.read_csv(r"c:\Users\DELL\Desktop\phasenet_picks_2208_11raw.csv")
df2 = pd.read_csv(r"D:\MyRepository\TranSeis\PredictionResults4QJ20228_11_raw\_outputs\picks_results_raw20228_11.csv")
print(df1.columns)
# === 2️⃣ 时间列解析为 UTC ===
df1["time"] = pd.to_datetime(df1["time"], utc=True, errors="coerce")
df2["phase_time"] = pd.to_datetime(df2["phase_time"], utc=True, errors="coerce")

# 删除空值
df1 = df1.dropna(subset=["time"])
df2 = df2.dropna(subset=["phase_time"])

# === 3️⃣ 提取台站编号，统一格式 ===
df1["station_id"] = df1["station_id"].str.extract(r'([A-Z]+\d+)')
df2["station_id"] = df2["station_id"].astype(str).str.strip()

# === 4️⃣ KDTree 最近邻匹配 ===
result_list = []

# 时间容差（秒）
TOLERANCE = 50000

for phase_type in ["P", "S"]:
    for station in df1["station_id"].dropna().unique():
        a = df1[(df1["station_id"] == station) & (df1["phase"] == phase_type)].copy()
        b = df2[(df2["station_id"] == station) & (df2["phase_type"] == phase_type)].copy()
        if len(a) == 0 or len(b) == 0:
            continue

        # 转为时间戳（秒）
        t1 = a["time"].view(np.int64) / 1e9
        t2 = b["phase_time"].view(np.int64) / 1e9

        # 建立 KDTree
        tree = cKDTree(t2.values.reshape(-1, 1))
        dist, idx = tree.query(t1.values.reshape(-1, 1), distance_upper_bound=TOLERANCE)

        # 如果超出范围（inf）则设为 NaN
        matched_amplitude = np.full(len(a), np.nan)
        valid_mask = np.isfinite(dist)
        matched_amplitude[valid_mask] = b["phase_amplitude"].values[idx[valid_mask]]

        # 添加匹配结果
        a["phase_amplitude"] = matched_amplitude
        result_list.append(a)

        print(f"✅ 匹配完成 {station} - {phase_type}：匹配 {valid_mask.sum()}/{len(a)} 条")

# === 5️⃣ 合并所有结果 ===
df_out = pd.concat(result_list, ignore_index=True)

# === 6️⃣ 选取输出列 ===
df_out = df_out[[
    "index", "station_id", "time", "start_time", "end_time",
    "probability", "phase", "phase_amplitude"
]]

# === 7️⃣ 保存结果 ===
output_path = r"c:\Users\DELL\Desktop\phasenet_picks_2208_11raw.csv"
df_out.to_csv(output_path, index=False)

print(f"\n✅ 已完成匹配并输出到：{output_path}")

C:\Users\DELL\AppData\Local\Temp\ipykernel_64696\2677772872.py:6: DtypeWarning: Columns (0,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv(r"c:\Users\DELL\Desktop\phasenet_picks_2208_11raw.csv")


Index(['index', 'station_id', 'time', 'start_time', 'end_time', 'probability',
       'phase', 'phase_amplitude'],
      dtype='object')


C:\Users\DELL\AppData\Local\Temp\ipykernel_64696\2677772872.py:35: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  t1 = a["time"].view(np.int64) / 1e9
C:\Users\DELL\AppData\Local\Temp\ipykernel_64696\2677772872.py:36: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  t2 = b["phase_time"].view(np.int64) / 1e9


✅ 匹配完成 QJ01 - P：匹配 1160/1160 条
✅ 匹配完成 QJ02 - P：匹配 1181/1181 条
✅ 匹配完成 QJ03 - P：匹配 1043/1043 条
✅ 匹配完成 QJ04 - P：匹配 3336/3336 条
✅ 匹配完成 QJ05 - P：匹配 3326/3326 条
✅ 匹配完成 QJ06 - P：匹配 1102/1102 条
✅ 匹配完成 QJ07 - P：匹配 1178/1178 条
✅ 匹配完成 QJ08 - P：匹配 1195/1195 条
✅ 匹配完成 QJ09 - P：匹配 3036/3036 条
✅ 匹配完成 QJ10 - P：匹配 1183/1183 条
✅ 匹配完成 QJ01 - S：匹配 1168/1168 条
✅ 匹配完成 QJ02 - S：匹配 1183/1183 条
✅ 匹配完成 QJ03 - S：匹配 1097/1097 条
✅ 匹配完成 QJ04 - S：匹配 3334/3334 条
✅ 匹配完成 QJ05 - S：匹配 3342/3342 条
✅ 匹配完成 QJ06 - S：匹配 1089/1089 条
✅ 匹配完成 QJ07 - S：匹配 1193/1193 条
✅ 匹配完成 QJ08 - S：匹配 1197/1197 条
✅ 匹配完成 QJ09 - S：匹配 3079/3079 条
✅ 匹配完成 QJ10 - S：匹配 1184/1184 条

✅ 已完成匹配并输出到：c:\Users\DELL\Desktop\phasenet_picks_2208_11raw.csv


### 重新找phase_amplitude

In [23]:
import os
import pandas as pd
import numpy as np
from obspy import read
from datetime import datetime, timedelta

mseed_dir = r"d:/mseed_merged/"

df = pd.read_csv(r"c:\Users\DELL\Desktop\eqt_picks_raw2212.csv")
df["start_time"] = pd.to_datetime(df["start_time"], format="mixed", errors="coerce")
df["end_time"]   = pd.to_datetime(df["end_time"],   format="mixed", errors="coerce")

import glob
import os

def find_mseed_file(station_id, start_time):
    """
    根据 start_time 精确定位该小时的 segment 文件，
    并忽略中间变化的编号（如 8480、8483）
    """

    # ① 小时起始时间
    hour_start = start_time.replace(minute=0, second=0, microsecond=0)
    hour_str = hour_start.strftime("%Y%m%d_%H0000")

    # ② 在该小时内的秒数偏移
    seconds_from_hour = (start_time - hour_start).total_seconds()

    # ③ segment 号（每 200 秒一个）
    seg_id = int(seconds_from_hour // 200)

    # ④ 使用通配符 * 忽略中间变化的编号
    #    例如：QJ.QJ08._centaur-3_*_20230114_150000.miniseed_segment_0.mseed
    pattern = (
        f"{station_id}_centaur-3_*_{hour_str}.miniseed_segment_{seg_id}.mseed"
    )

    # 在目录中匹配文件
    matches = glob.glob(os.path.join(mseed_dir, pattern))

    if len(matches) == 0:
        print(f"⚠ 未找到文件: pattern = {pattern}")
        return None

    # 正常情况下一个小时的某个 segment 应该只有一个文件
    return matches[0]


# --- 遍历每一行 pick ---
for idx, row in df.iterrows():
    station = row["station_id"]
    t0 = row["start_time"]
    t1 = row["end_time"]

    mseed_file = find_mseed_file(station, t0)
    if mseed_file is None:
        print(f"❗ 未找到 mseed 文件: {station}, {t0}")
        continue

        # ====== 读取并合并三通道 ======
    st = read(mseed_file)

    # 按 channel 排序（确保顺序一致）
    st.sort(['channel'])

    # dataset: (samples, 3)
    dataset = np.column_stack([tr.data for tr in st])
    # ===============================

    # 振幅 envelope-like : max(|value|) per sample
    amp = np.max(np.abs(dataset), axis=-1)   # (samples,)

    # 采样率
    sr = st[0].stats.sampling_rate
    t_start = st[0].stats.starttime.datetime

    # 切片索引
    idx0 = int((t0 - t_start).total_seconds() * sr)
    idx1 = int((t1 - t_start).total_seconds() * sr)

    # 防止越界
    idx0 = max(0, idx0)
    idx1 = min(len(amp)-1, idx1)

    subset = amp[idx0:idx1+1]

    if len(subset) > 0:
        df.loc[idx, "phase_amplitude"] = float(np.max(subset))
    else:
        print(f"⚠️ subset 为空: {station}, {t0}")
    

# 保存
df.to_csv(r"c:\Users\DELL\Desktop\eqt_picks_2212raw_withamp_duibi.csv", index=False)

print("✅ 已完成更新 phase_amplitude")

✅ 已完成更新 phase_amplitude
